# Stage 7 — MFCC/SVM baseline fit (step 2)

This notebook runs the official baseline fit from an exact clean commit: it builds the MFCC feature caches for FoR train, val-A and val-B, fits the baseline (selection on val-A, D29 threshold on val-B; D9, D37) and exports the evidence. Every step runs twice and the outputs must be byte-identical. Run it only after the implementation commit is reviewed and merged.

## 1. Choose transfer mode and check out the exact implementation commit

Replace `IMPLEMENTATION_COMMIT` with the full approved commit. Leave `TRANSFER_MODE` as `manual` when Colab cannot mount Drive, or change it to `drive` when mounting works. In `drive` mode Drive is mounted before anything else. The checkout must remain clean.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

TRANSFER_MODE = 'manual'  # 'manual' or 'drive'
IMPLEMENTATION_COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'
if TRANSFER_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif TRANSFER_MODE != 'manual':
    raise ValueError("TRANSFER_MODE must be 'manual' or 'drive'")
REPO = Path('/content/voice-spoofing-detection')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', 'https://github.com/kahlanzzi/voice-spoofing-detection.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', IMPLEMENTATION_COMMIT], cwd=REPO, check=True)
assert not subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, check=True, capture_output=True, text=True).stdout
os.chdir(REPO)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 2. Provide and extract the pinned FoR archive

In `manual` mode, first use Colab's Files panel to upload `for_2sec_ff8c82c7.tar` and `for_2sec_REVISION.txt` into `/content/`. In `drive` mode, the files are copied from the shared raw folder. This step verifies the revision marker and extracts only the approved FoR archive.

In [ ]:
import tarfile

if TRANSFER_MODE == 'drive':
    DRIVE_ROOT = Path('/content/drive/MyDrive/voice-spoofing-detection')
    ARCHIVE = DRIVE_ROOT / 'raw' / 'for_2sec_ff8c82c7.tar'
    REVISION_FILE = DRIVE_ROOT / 'raw' / 'for_2sec_REVISION.txt'
else:
    ARCHIVE = Path('/content/for_2sec_ff8c82c7.tar')
    REVISION_FILE = Path('/content/for_2sec_REVISION.txt')
    missing = [str(path) for path in (ARCHIVE, REVISION_FILE) if not path.is_file()]
    if missing:
        raise FileNotFoundError('Upload these files with the Colab Files panel: ' + ', '.join(missing))
EXPECTED_REVISION = 'ff8c82c79e7bbefef811941bac9775c9328e9055'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
LOCAL_ARCHIVE = Path('/content/data/for_2sec_ff8c82c7.tar')
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
if ARCHIVE.resolve() != LOCAL_ARCHIVE.resolve():
    shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
print('FoR archive extracted to', AUDIO_ROOT)

## 3. Run the test suite

In [ ]:
subprocess.run(['python', '-m', 'pytest', '-q'], cwd=REPO, check=True)

## 4. Build the three feature caches twice and compare the bytes

Each split (train, val-A, val-B) is cached into `cache_a` and again into `cache_b`. Every sample hash is checked before decoding. The two builds must give identical npz and JSON bytes.

In [ ]:
import hashlib, json

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

WORK = Path('/content/stage7')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
SPLITS = ['train', 'val-A', 'val-B']
CACHE_A, CACHE_B = WORK / 'cache_a', WORK / 'cache_b'
for cache in (CACHE_A, CACHE_B):
    for split in SPLITS:
        subprocess.run([
            'python', '-m', 'src.features.mfcc_cache',
            '--root', str(AUDIO_ROOT), '--manifest', str(MANIFEST), '--profile', str(PROFILE),
            '--split', split, '--out', str(cache / split),
        ], cwd=REPO, check=True, stdout=subprocess.DEVNULL)
cache_hashes = {}
for split in SPLITS:
    for name in (f'{split}.npz', f'{split}.json'):
        assert (CACHE_A / split / name).read_bytes() == (CACHE_B / split / name).read_bytes(), name
        cache_hashes[name] = sha256(CACHE_A / split / name)
determinism = {
    'code_commit': IMPLEMENTATION_COMMIT,
    'caches_byte_identical': True,
    'cache_sha256': cache_hashes,
}
DETERMINISM = WORK / 'determinism.json'
DETERMINISM.write_text(json.dumps(determinism, indent=2, sort_keys=True) + '\n', encoding='utf-8')
determinism

## 5. Fit the baseline twice and compare the bytes

Both runs read `cache_a`. Selection uses val-A only and the threshold val-B only (D9, D29, D37). All reports and both artifact files must be byte-identical, with the same `artifact_sha256`.

In [ ]:
RUN_A, RUN_B = WORK / 'run_a', WORK / 'run_b'
for output in (RUN_A, RUN_B):
    subprocess.run([
        'python', '-m', 'src.tuning.baseline_run',
        '--cache-dir', str(CACHE_A), '--profile', str(PROFILE),
        '--config', str(REPO / 'configs/baseline.yaml'), '--manifest', str(MANIFEST),
        '--out', str(output), '--version', 'baseline-v1',
    ], cwd=REPO, check=True)
REPORTS = ['selection_report.json', 'threshold_report.json', 'run_report.json', 'subgroup_report.json']
files_a = sorted(p.relative_to(RUN_A).as_posix() for p in RUN_A.rglob('*') if p.is_file())
files_b = sorted(p.relative_to(RUN_B).as_posix() for p in RUN_B.rglob('*') if p.is_file())
assert files_a == files_b == sorted(REPORTS + ['artifact/baseline.json', 'artifact/baseline.npz']), files_a
for rel in files_a:
    assert (RUN_A / rel).read_bytes() == (RUN_B / rel).read_bytes(), rel
report_a = json.loads((RUN_A / 'run_report.json').read_text(encoding='utf-8'))
report_b = json.loads((RUN_B / 'run_report.json').read_text(encoding='utf-8'))
assert report_a['artifact_sha256'] == report_b['artifact_sha256']
determinism.update({
    'runs_byte_identical': True,
    'run_sha256': {rel: sha256(RUN_A / rel) for rel in files_a},
    'artifact_sha256': report_a['artifact_sha256'],
})
DETERMINISM.write_text(json.dumps(determinism, indent=2, sort_keys=True) + '\n', encoding='utf-8')
determinism

## 6. Export the evidence and the artifact

Evidence (the three cache JSON files, the four reports and `determinism.json`) goes to Drive `processed/metadata/stage7/`. The artifact folder goes to Drive `checkpoints/baseline/baseline-v1/` only; an existing artifact folder is never overwritten. In `manual` mode, two zips are downloaded: upload the evidence files to `processed/metadata/stage7/` and the artifact files to `checkpoints/baseline/baseline-v1/` using the normal Drive website. These are candidate records until human review.

In [ ]:
EVIDENCE = WORK / 'evidence'
EVIDENCE.mkdir()
for split in SPLITS:
    shutil.copy2(CACHE_A / split / f'{split}.json', EVIDENCE / f'{split}.json')
for name in REPORTS:
    shutil.copy2(RUN_A / name, EVIDENCE / name)
shutil.copy2(DETERMINISM, EVIDENCE / 'determinism.json')
if TRANSFER_MODE == 'drive':
    evidence_dest = DRIVE_ROOT / 'processed' / 'metadata' / 'stage7'
    artifact_dest = DRIVE_ROOT / 'checkpoints' / 'baseline' / 'baseline-v1'
    if artifact_dest.exists():
        raise FileExistsError(f'{artifact_dest} already exists; a frozen artifact is never overwritten')
    evidence_dest.mkdir(parents=True, exist_ok=True)
    for path in EVIDENCE.iterdir():
        shutil.copy2(path, evidence_dest / path.name)
    shutil.copytree(RUN_A / 'artifact', artifact_dest)
    print('Evidence copied to', evidence_dest)
    print('Artifact copied to', artifact_dest)
else:
    from google.colab import files
    evidence_zip = Path(shutil.make_archive(str(WORK / 'stage7_evidence'), 'zip', root_dir=EVIDENCE))
    artifact_zip = Path(shutil.make_archive(str(WORK / 'baseline-v1_artifact'), 'zip', root_dir=RUN_A / 'artifact'))
    files.download(str(evidence_zip))
    files.download(str(artifact_zip))
    print('Downloaded', evidence_zip.name, 'for Drive processed/metadata/stage7/')
    print('Downloaded', artifact_zip.name, 'for Drive checkpoints/baseline/baseline-v1/')
print('artifact_sha256:', report_a['artifact_sha256'])
print('Reminder: the baseline artifact is never committed to Git; only the reports are committed after review.')